In [2]:
import pandas as pd
df = pd.read_csv("/content/fake_job_postings.csv", engine='python')
print(df.head())
print(df.columns)
print(df.shape)

print(df["fraudulent"].value_counts())
print(df["fraudulent"].value_counts(normalize=True)*100)
print(df[['company_profile', 'description', 'requirements']].isnull().sum())
df["text"] = (df['title'].fillna('') + ' ' + df['company_profile'].fillna('') + ' ' + df['description'].fillna('') + ' ' + df['requirements'].fillna(''))
print(df['text'].iloc[0][:300])

   job_id                                      title            location  \
0       1                           Marketing Intern    US, NY, New York   
1       2  Customer Service - Cloud Video Production      NZ, , Auckland   
2       3    Commissioning Machinery Assistant (CMA)       US, IA, Wever   
3       4          Account Executive - Washington DC  US, DC, Washington   
4       5                        Bill Review Manager  US, FL, Fort Worth   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   
2        NaN          NaN  Valor Services provides Workforce Solutions th...   
3      Sales          NaN  Our passion for improving quality of life thro...   
4        NaN          NaN  SpotSource Solutions LLC is a Global Human Cap...   

                                         description  \
0  Foo

In [6]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix
import numpy as np

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
df['clean_text'] = df['text'].apply(clean_text)
df[['text','clean_text']].sample(3)

X = df['clean_text']
y = df['fraudulent']
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )
print(X_train.shape, X_test.shape)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(X_train_tfidf.shape)

model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
top_fraud_words = pd.DataFrame({ 'word': feature_names, 'weight': coefficients }).sort_values('weight', ascending=False).head(20)
print(top_fraud_words)

def explain_prediction(index, X_test_tfidf, X_test, feature_names, coefficients, top_n=5):
    row = X_test_tfidf[index].toarray()[0]
    word_scores = row * coefficients
    top_indices = word_scores.argsort()[-top_n:][::-1]
    top_words = []
    for i in top_indices:
        if row[i] > 0:
            top_words.append((feature_names[i], round(word_scores[i], 3)))
    return top_words
sample_index = 0
print(explain_prediction(sample_index, X_test_tfidf, X_test, feature_names, coefficients))
print("Actual label:", y_test.iloc[0])
print("Predicted label:", y_pred[0])
print("Posting text:", X_test.iloc[0][:300])

test_df = pd.DataFrame({ 'text': X_test.values, 'actual': y_test.values, 'predicted': y_pred }).reset_index(drop=True)
fraud_correctly_caught = test_df[(test_df['actual']==1) & (test_df['predicted']==1)]
print(fraud_correctly_caught.head(3))

demo_index = 84
print("Actual:", y_test.iloc[demo_index], "Predicted:", y_pred[demo_index])
print("Text:", X_test.iloc[demo_index][:300])
print("Why flagged:", explain_prediction(demo_index, X_test_tfidf, X_test, feature_names, coefficients))

all_tfidf = vectorizer.transform(df['clean_text'])
df['fraud_probability'] = model.predict_proba(all_tfidf)[:, 1]
df['predicted_fraud'] = model.predict(all_tfidf)
print(df[['title','fraudulent','fraud_probability','predicted_fraud']].head())

full_tfidf = vectorizer.transform(df['clean_text'])
def get_top_words(index, tfidf_matrix, feature_names, coefficients, top_n=5):
  row = tfidf_matrix[index].toarray()[0]
  word_scores = row * coefficients
  top_indices = word_scores.argsort()[-top_n:][::-1]
  words = [feature_names[i] for i in top_indices if row[i] > 0]
  return ', '.join(words)
df['flag_reasons'] = [get_top_words(i, full_tfidf, feature_names, coefficients)  for i in range(len(df))]
print(df[['title','fraud_probability','flag_reasons']].sample(5))

df['has_salary'] = df['salary_range'].notna().astype(int)
export_cols = ['job_id', 'title', 'location', 'industry', 'employment_type', 'has_company_logo', 'telecommuting', 'has_salary', 'fraudulent', 'predicted_fraud', 'fraud_probability', 'flag_reasons']
df[export_cols].to_csv('scored_postings.csv', index=False)
print("Saved:", df[export_cols].shape)

(14304,) (3576,)
(14304, 5000)
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3403
           1       0.63      0.90      0.74       173

    accuracy                           0.97      3576
   macro avg       0.81      0.93      0.86      3576
weighted avg       0.98      0.97      0.97      3576

[[3311   92]
 [  18  155]]
                word    weight
2646            link  4.836792
291         aptitude  4.357339
1599           entry  3.954486
2907           money  3.857570
1843       financing  3.559596
3304           phone  3.358152
1442            earn  3.267581
798            clerk  3.122817
104   administrative  3.040804
1563     engineering  2.955677
1994             gas  2.895111
722               cf  2.857094
3826    requirements  2.824571
4395          subsea  2.813310
683             cash  2.759963
3064             oil  2.736067
2179        hospital  2.735546
3061        offshore  2.685886
40        accountant  2.66